In [ ]:
# Celda 1: imports y carga del modelo
# Carga modelo_clasificacion2 (EfficientNetB0 multi-clase con softmax,
# entrenado en cnn_damage_multiclass_training.ipynb) y la lista de clases
# en el orden de las salidas del modelo. El preprocess de EfficientNet va
# dentro del modelo, asi que las imagenes se pasan en 0-255 sin normalizar.

import os
import json
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

MODEL_PATH = "../models/modelo_clasificacion2.keras"

model = tf.keras.models.load_model(MODEL_PATH)

with open("../models/modelo_clasificacion2_clases.json", encoding="utf-8") as f:
    FINAL_CLASSES = json.load(f)

print(f"Modelo cargado: {MODEL_PATH} | Clases: {len(FINAL_CLASSES)}")

In [ ]:
# Celda 2: configuracion
# USAR_DATASET = True toma N_POR_CLASE imagenes al azar de cada clase del
# dataset ya descargado por cnn_damage_classification_ETL_final.ipynb
# (data_clasificacion/dataset_final.csv), asi se ve la clase real al lado
# de la predicha. Cada vez que se corre la celda 4 salen imagenes distintas.
# USAR_DATASET = False prueba todas las imagenes de FOLDER_PATH (fotos
# propias, sin clase real).
# IMG_SIZE tiene que ser el mismo del entrenamiento (224x224) y TOP_K es
# cuantas clases mostrar por imagen.

IMG_SIZE = (224, 224)
TOP_K = 3

USAR_DATASET = True
DATASET_CSV = "data_clasificacion/dataset_final.csv"
N_POR_CLASE = 3

FOLDER_PATH = "prueba"

In [ ]:
# Celda 3: verificar el dataset del ETL final
# Solo si USAR_DATASET. Chequea que este lo que genera
# cnn_damage_classification_ETL_final.ipynb y que coincida con el modelo:
#   - dataset_final.csv existe y tiene las columnas image_path y label
#   - las clases del csv son las mismas 11 del modelo
#   - grupos_final.json (al lado del csv) es igual a modelo_clasificacion2_grupos.json
#   - las imagenes del csv estan en disco
# Si algo falla corta con un error que dice que falta; si no, muestra
# cuantas imagenes hay por clase.
# Depende de: USAR_DATASET, DATASET_CSV (celda 2), FINAL_CLASSES (celda 1)

if not USAR_DATASET:
    print("USAR_DATASET = False: no se usa el dataset, no hay nada que verificar")
else:
    if not os.path.exists(DATASET_CSV):
        raise FileNotFoundError(f"No existe {DATASET_CSV}: correr antes cnn_damage_classification_ETL_final.ipynb")

    df_check = pd.read_csv(DATASET_CSV)
    problemas = []

    faltan_columnas = {"image_path", "label"} - set(df_check.columns)
    if faltan_columnas:
        problemas.append(f"al csv le faltan las columnas {sorted(faltan_columnas)}")
    else:
        clases_csv = set(df_check["label"])
        if clases_csv - set(FINAL_CLASSES):
            problemas.append(f"clases en el csv que el modelo no tiene: {sorted(clases_csv - set(FINAL_CLASSES))}")
        if set(FINAL_CLASSES) - clases_csv:
            problemas.append(f"clases del modelo que no estan en el csv: {sorted(set(FINAL_CLASSES) - clases_csv)}")

        no_estan = [r for r in df_check["image_path"] if not os.path.exists(r)]
        if no_estan:
            problemas.append(f"{len(no_estan)} de {len(df_check)} imagenes no estan en disco (ej: {no_estan[0]})")

    grupos_path = os.path.join(os.path.dirname(DATASET_CSV), "grupos_final.json")
    with open("../models/modelo_clasificacion2_grupos.json", encoding="utf-8") as f:
        grupos_modelo = json.load(f)
    if not os.path.exists(grupos_path):
        problemas.append(f"no existe {grupos_path}")
    else:
        with open(grupos_path, encoding="utf-8") as f:
            if json.load(f) != grupos_modelo:
                problemas.append("grupos_final.json no coincide con los grupos del modelo (el dataset es de otro ETL)")

    if problemas:
        raise ValueError("El dataset no sirve para este modelo:\n  - " + "\n  - ".join(problemas))

    print(f"Dataset OK: {len(df_check)} imagenes, todas en disco, mismas clases y grupos que el modelo\n")
    print(df_check["label"].value_counts().reindex(FINAL_CLASSES).to_string())

In [ ]:
# Celda 4: funcion de prediccion
# Lee la imagen igual que en el pipeline de entrenamiento (decode_image +
# resize) y devuelve todas las clases ordenadas de mayor a menor
# probabilidad. Como es softmax, las probabilidades suman 1 y la
# prediccion es la primera de la lista.
# Depende de: model, FINAL_CLASSES, IMG_SIZE

def predecir_dano(ruta_imagen):
    img = tf.io.read_file(ruta_imagen)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, IMG_SIZE)
    probs = model.predict(tf.expand_dims(img, 0), verbose=0)[0]
    orden = probs.argsort()[::-1]
    return img, [(FINAL_CLASSES[i], float(probs[i])) for i in orden]

In [ ]:
# Celda 5: elegir las imagenes a probar
# Con USAR_DATASET rearma el mismo split train/val/test del entrenamiento
# (mismo csv, mismos train_test_split con random_state=42) y elige al azar
# N_POR_CLASE imagenes por clase del test, que son imagenes que el modelo
# no vio al entrenar. Si no, toma todas las imagenes de FOLDER_PATH.
# Queda un df a_probar con image_path y label (None si no hay clase real).
# Depende de: USAR_DATASET, DATASET_CSV, N_POR_CLASE, FOLDER_PATH (celda 2)

if USAR_DATASET:
    df = pd.read_csv(DATASET_CSV)
    _, resto_df = train_test_split(df, test_size=0.30, stratify=df["label"], random_state=42)
    _, test_df = train_test_split(resto_df, test_size=0.50, stratify=resto_df["label"], random_state=42)
    a_probar = (test_df.groupby("label", group_keys=False)
                .sample(n=N_POR_CLASE)
                .set_index("label").loc[FINAL_CLASSES].reset_index())
    print(f"test: {len(test_df)} imagenes | se prueban {len(a_probar)} ({N_POR_CLASE} por clase)")
else:
    valid_extensions = (".jpg", ".jpeg", ".png")
    archivos = sorted(f for f in os.listdir(FOLDER_PATH) if f.lower().endswith(valid_extensions))
    a_probar = pd.DataFrame({"image_path": [os.path.join(FOLDER_PATH, f) for f in archivos], "label": None})
    print(f"Se encontraron {len(a_probar)} imagenes en '{FOLDER_PATH}'")

In [ ]:
# Celda 6: prediccion
# Para cada imagen muestra la foto con la clase predicha en el titulo
# (y la real si se conoce: verde si acerto, rojo si no) y al lado el
# grafico de barras de las TOP_K clases mas probables. Si hay clase real,
# al final imprime cuantas acerto.
# Depende de: predecir_dano (celda 4), a_probar (celda 5), TOP_K (celda 2)

aciertos = 0
for ruta, real in zip(a_probar["image_path"], a_probar["label"]):
    img, ranking = predecir_dano(ruta)
    clase, prob = ranking[0]
    top = ranking[:TOP_K]

    fig, (ax_img, ax_bar) = plt.subplots(1, 2, figsize=(11, 4), gridspec_kw={"width_ratios": [1.2, 1]})
    ax_img.imshow(img.numpy().astype("uint8"))
    ax_img.axis("off")
    if real is None:
        ax_img.set_title(f"{os.path.basename(ruta)}\npred: {clase} ({prob:.2%})")
    else:
        aciertos += clase == real
        color = "green" if clase == real else "red"
        ax_img.set_title(f"real: {real}\npred: {clase} ({prob:.2%})", color=color)

    ax_bar.barh([c for c, _ in top][::-1], [p for _, p in top][::-1])
    ax_bar.set_xlim(0, 1)
    ax_bar.set_xlabel("probabilidad")
    plt.tight_layout()
    plt.show()

    print(f"{os.path.basename(ruta)}: " + ", ".join(f"{c} ({p:.2%})" for c, p in top) + "\n")

if a_probar["label"].notna().all():
    print(f"Acerto {aciertos} de {len(a_probar)} ({aciertos / len(a_probar):.0%})")